In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('Task 3 and 4_Loan_Data.csv')

# Group by FICO score — many borrowers share the same score
# For each unique score: how many total (n) and how many defaulted (k)
fico_data = (df.groupby('fico_score')['default']
               .agg(n='count', k='sum')
               .reset_index()
               .sort_values('fico_score')
               .reset_index(drop=True))

print(fico_data.head(10))
print(f"\nUnique FICO values: {len(fico_data)}")
print(f"FICO range: {fico_data['fico_score'].min()} – {fico_data['fico_score'].max()}")

   fico_score  n  k
0         408  1  0
1         409  1  1
2         418  1  1
3         425  1  1
4         438  1  1
5         440  1  1
6         441  1  1
7         444  1  0
8         447  1  0
9         449  1  1

Unique FICO values: 374
FICO range: 408 – 850


In [2]:
n_scores = len(fico_data)

prefix_n = np.zeros(n_scores + 1, dtype=int)
prefix_k = np.zeros(n_scores + 1, dtype=int)

for i in range(n_scores):
    prefix_n[i+1] = prefix_n[i] + fico_data['n'].iloc[i]
    prefix_k[i+1] = prefix_k[i] + fico_data['k'].iloc[i]

print(f"Total borrowers : {prefix_n[-1]}")
print(f"Total defaults  : {prefix_k[-1]}")

Total borrowers : 10000
Total defaults  : 1851


In [3]:
def bucket_ll(l, r):
    """Log-likelihood for bucket covering FICO indices l..r inclusive"""
    n = prefix_n[r+1] - prefix_n[l]
    k = prefix_k[r+1] - prefix_k[l]

    # Edge cases: if no defaults or all default, log(0) undefined
    # These buckets are perfectly homogeneous — contribute 0
    if n == 0 or k == 0 or k == n:
        return 0.0

    p = k / n
    return k * np.log(p) + (n - k) * np.log(1 - p)

In [4]:
def optimal_buckets(num_buckets):
    N = n_scores  # 374 unique FICO values
    
    # Initialize DP tables with -infinity
    dp    = np.full((N, num_buckets + 1), -np.inf)
    split = np.zeros((N, num_buckets + 1), dtype=int)

    # Base case: j=1, one bucket from 0 to i
    for i in range(N):
        dp[i][1] = bucket_ll(0, i)

    # Fill DP for j=2 to num_buckets
    for j in range(2, num_buckets + 1):
        for i in range(j - 1, N):           # need at least j indices for j buckets
            for m in range(j - 2, i):       # last bucket is m+1..i
                val = dp[m][j-1] + bucket_ll(m + 1, i)
                if val > dp[i][j]:
                    dp[i][j]    = val
                    split[i][j] = m + 1     # last bucket starts here

    # Reconstruct boundaries by backtracking through split table
    boundaries = []
    i = N - 1
    j = num_buckets
    while j > 1:
        start = split[i][j]
        boundaries.append(fico_data['fico_score'].iloc[start])
        i = start - 1
        j -= 1
    
    boundaries.reverse()
    return boundaries

In [5]:
num_buckets = 5
bounds = optimal_buckets(num_buckets)

# Add the outer edges
all_bounds = [fico_data['fico_score'].min()] + bounds + [fico_data['fico_score'].max()]

print(f"Optimal boundaries for {num_buckets} buckets:")
for i in range(len(all_bounds) - 1):
    lo = all_bounds[i]
    hi = all_bounds[i+1]
    
    # Get stats for this bucket
    mask = (df['fico_score'] >= lo) & (df['fico_score'] < hi)
    if i == len(all_bounds) - 2:  # last bucket inclusive on right
        mask = (df['fico_score'] >= lo) & (df['fico_score'] <= hi)
    
    n_b = mask.sum()
    k_b = df[mask]['default'].sum()
    pd_b = k_b / n_b if n_b > 0 else 0
    
    print(f"  Bucket {i+1}: FICO {lo:3d} – {hi:3d} | "
          f"n={n_b:5d} | defaults={k_b:4d} | PD={pd_b:.3f}")

Optimal boundaries for 5 buckets:
  Bucket 1: FICO 408 – 521 | n=  301 | defaults= 199 | PD=0.661
  Bucket 2: FICO 521 – 581 | n= 1407 | defaults= 536 | PD=0.381
  Bucket 3: FICO 581 – 641 | n= 3438 | defaults= 703 | PD=0.204
  Bucket 4: FICO 641 – 697 | n= 3197 | defaults= 336 | PD=0.105
  Bucket 5: FICO 697 – 850 | n= 1657 | defaults=  77 | PD=0.046


In [6]:
def fico_to_rating(fico_score, num_buckets=5):
    """
    Returns rating 1 (best) to num_buckets (worst).
    Lower rating = higher FICO = lower default risk.
    """
    bounds = optimal_buckets(num_buckets)
    all_bounds = ([fico_data['fico_score'].min()] 
                  + bounds + 
                  [fico_data['fico_score'].max()])
    
    for i in range(len(all_bounds) - 1):
        lo = all_bounds[i]
        hi = all_bounds[i + 1]
        if fico_score <= hi:
            # Rating 1 = best (highest FICO bucket), n = worst
            return num_buckets - i

    return 1  # max FICO edge case

# Test it
for score in [420, 540, 610, 670, 750]:
    print(f"FICO {score} → Rating {fico_to_rating(score)}")

FICO 420 → Rating 5
FICO 540 → Rating 4
FICO 610 → Rating 3
FICO 670 → Rating 2
FICO 750 → Rating 1
